# NRMS on Steam — three text conditions

Upload these two files:
- `steam_prepared_outputs.zip` — your prepared data
- `tag_descriptions_deepseek.json` — your 433 DeepSeek tag descriptions



## 1. Setup and uploads

In [1]:
import os, sys, time, json, random, gc
from pathlib import Path

# Confirm GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime -> Change runtime type -> GPU (T4) before continuing.')


CUDA available: True
Device: Tesla T4


In [2]:
# Install missing pieces (transformers usually already present, but safe to re-pin)
%pip install -q transformers==4.44.2 scikit-learn pyarrow pandas tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 103.8 MB/s eta 0:00:00


In [3]:
# Upload BOTH files in this single dialog: steam_prepared_outputs.zip + tag_descriptions_deepseek.json
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import files
    print('Upload steam_prepared_outputs.zip AND tag_descriptions_deepseek.json (multi-select OK):')
    up = files.upload()
    print('Got files:', list(up.keys()))
else:
    print('Not on Colab — place both files in the working directory.')


Upload steam_prepared_outputs.zip AND tag_descriptions_deepseek.json (multi-select OK):


Got files: []


In [5]:
# Unzip + locate the prepared-data folder
import zipfile, shutil

WORK = Path('/content') if IN_COLAB else Path('.')
DATA_DIR = WORK / 'steam_prepared_outputs'

if not DATA_DIR.exists():
    zip_path = WORK / 'steam_prepared_outputs.zip'
    assert zip_path.exists(), f'Missing {zip_path} — upload it first.'
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(WORK)
    # Some zips wrap everything in an inner folder; find the one with items.parquet
    if not (DATA_DIR / 'items.parquet').exists():
        for p in WORK.rglob('items.parquet'):
            DATA_DIR = p.parent
            break

print('DATA_DIR =', DATA_DIR)
for f in sorted(DATA_DIR.iterdir()):
    print(f'  {f.name:35s} {f.stat().st_size/1024:8.1f} KB')

# Make sure tag descriptions are available
TAG_DESC_PATH = WORK / 'tag_descriptions_deepseek.json'
if not TAG_DESC_PATH.exists():
    # maybe it was unzipped along with the data
    alt = DATA_DIR / 'tag_descriptions_deepseek.json'
    if alt.exists():
        TAG_DESC_PATH = alt
assert TAG_DESC_PATH.exists(), 'tag_descriptions_deepseek.json missing — re-upload it.'
print('tag descriptions:', TAG_DESC_PATH)


DATA_DIR = /content
  .config                                  4.0 KB
  interactions_clean.parquet            1123.3 KB
  items.csv                             2613.0 KB
  items.parquet                         1333.2 KB
  manifest.json                            0.6 KB
  sample_data                              4.0 KB
  samples_test.jsonl                   62450.8 KB
  samples_test.parquet                  1286.1 KB
  samples_train.jsonl                  84372.6 KB
  samples_train.parquet                 2811.8 KB
  samples_val.jsonl                    60374.5 KB
  samples_val.parquet                   1237.9 KB
  steam_prepared_outputs.zip           16426.1 KB
  summary_stats.csv                        0.3 KB
  tag_descriptions_deepseek.json         162.2 KB
  tags_unique.csv                          6.0 KB
tag descriptions: /content/tag_descriptions_deepseek.json


## 2. Load items, samples, tag descriptions

In [6]:
import pandas as pd
import numpy as np

items = pd.read_parquet(DATA_DIR / 'items.parquet')
train_samples = pd.read_parquet(DATA_DIR / 'samples_train.parquet')
val_samples   = pd.read_parquet(DATA_DIR / 'samples_val.parquet')
test_samples  = pd.read_parquet(DATA_DIR / 'samples_test.parquet')

with open(TAG_DESC_PATH, 'r', encoding='utf-8') as f:
    tag_descriptions = json.load(f)

print(f'items         : {len(items):,} games')
print(f'train samples : {len(train_samples):,} rows')
print(f'val   samples : {len(val_samples):,} rows')
print(f'test  samples : {len(test_samples):,} rows')
print(f'tag desc      : {len(tag_descriptions)} tags')


items         : 3,825 games
train samples : 519,002 rows
val   samples : 303,535 rows
test  samples : 294,159 rows
tag desc      : 433 tags


## 0. Generate SHORT tag descriptions



In [7]:
# ============================================================
# Generate SHORT tag descriptions via DeepSeek (run once)
# ============================================================
import os, json, time, sys
from pathlib import Path

# --- Where to load existing tags from ---
# After cell 7 runs, `items` is in memory. If we're running this BEFORE cell 7,
# we need to pull the unique tags directly from the prepared zip.
import zipfile
OUT_PATH = Path('tag_descriptions_short.json')

if OUT_PATH.exists():
    print(f'{OUT_PATH} already exists — delete it to regenerate.')
    short_descs_preview = json.load(open(OUT_PATH))
    print(f'  contains {len(short_descs_preview)} short descriptions')
else:
    # --- Find the unique tags. Prefer the in-memory `items` if available, else dig into the zip. ---
    tags_to_describe = None
    try:
        # If section 2 already ran, `items` is in memory
        all_tags = set()
        for tlist in items['tags']:
            if isinstance(tlist, (list, tuple)) or hasattr(tlist, 'tolist'):
                all_tags.update(tlist if isinstance(tlist, list) else list(tlist))
        tags_to_describe = sorted(all_tags)
        print(f'Got {len(tags_to_describe)} unique tags from in-memory items.')
    except NameError:
        # Section 2 hasn't run yet — load the items parquet directly
        print('items not in memory; reading from prepared zip...')
        import pandas as pd
        from io import BytesIO
        # the zip is uploaded by cell 4; if it's not extracted yet, do it now
        for guess in ['/content/steam_prepared_outputs.zip', 'steam_prepared_outputs.zip']:
            if Path(guess).exists():
                with zipfile.ZipFile(guess) as zf:
                    inames = zf.namelist()
                    items_name = [n for n in inames if n.endswith('items.parquet')][0]
                    with zf.open(items_name) as f:
                        items_tmp = pd.read_parquet(BytesIO(f.read()))
                break
        else:
            raise FileNotFoundError('steam_prepared_outputs.zip not found — run cell 4 first.')
        all_tags = set()
        for tlist in items_tmp['tags']:
            if isinstance(tlist, (list, tuple)) or hasattr(tlist, 'tolist'):
                all_tags.update(tlist if isinstance(tlist, list) else list(tlist))
        tags_to_describe = sorted(all_tags)
        print(f'Got {len(tags_to_describe)} unique tags from items.parquet.')

    # --- DeepSeek setup ---
    try:
        from openai import OpenAI
    except ImportError:
        import subprocess; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai'])
        from openai import OpenAI

    api_key = os.environ.get('DEEPSEEK_API_KEY')
    if not api_key:
        from getpass import getpass
        api_key = getpass('DeepSeek API key (or paste literal): ').strip()
        os.environ['DEEPSEEK_API_KEY'] = api_key

    client = OpenAI(api_key=api_key, base_url='https://api.deepseek.com')

    # --- The prompt: short, keyword-front-loaded, no filler ---
    SYSTEM_PROMPT = (
        'You write very short descriptors for Steam game tags. '
        'Given a tag, reply with at most 12 words describing what the tag means for gameplay. '
        'Start with concrete genre keywords (not articles or filler). '
        'Do not repeat the tag name. Do not use the phrase "this tag" or "refers to". '
        'Output the description only, no extra commentary, no quotation marks, no formatting.'
    )
    EXAMPLES = [
        ('Souls-like',     'punishing combat, stamina management, dying repeatedly, recovering souls, deliberate exploration.'),
        ('Roguelike',      'procedural runs, permadeath, randomized loot, escalating difficulty, build experimentation.'),
        ('Visual Novel',   'branching narrative, character portraits, dialogue-heavy, romance or mystery, choice-driven story.'),
        ('JRPG',           'turn-based party combat, anime aesthetics, leveling, story-driven exploration of fantasy worlds.'),
    ]

    def describe_short(tag: str, retries: int = 3) -> str:
        msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}]
        for tg, ex in EXAMPLES:
            msgs.append({'role': 'user',      'content': tg})
            msgs.append({'role': 'assistant', 'content': ex})
        msgs.append({'role': 'user', 'content': tag})
        for attempt in range(retries):
            try:
                r = client.chat.completions.create(
                    model='deepseek-chat', messages=msgs,
                    temperature=0.2, max_tokens=40,
                )
                text = r.choices[0].message.content.strip()
                # Truncate hard to ~15 words just in case
                words = text.split()
                if len(words) > 15:
                    text = ' '.join(words[:15])
                # Strip trailing period/quotes/asterisks
                text = text.strip('."\'* ')
                return text
            except Exception as e:
                print(f'   [retry {attempt+1}] {tag}: {e}')
                time.sleep(2 * (attempt + 1))
        return tag.lower()   # safe fallback

    # --- Generate with checkpointing ---
    short_descs = {}
    tmp_path = OUT_PATH.with_suffix('.partial.json')
    if tmp_path.exists():
        short_descs = json.load(open(tmp_path))
        print(f'Resuming with {len(short_descs)} already-done descriptions.')

    try:
        from tqdm.auto import tqdm
        pbar = tqdm(tags_to_describe, desc='DeepSeek (short)')
    except ImportError:
        pbar = tags_to_describe
    for i, tag in enumerate(pbar, 1):
        if tag in short_descs:
            continue
        short_descs[tag] = describe_short(tag)
        if i % 10 == 0:
            json.dump(short_descs, open(tmp_path, 'w'), indent=2, ensure_ascii=False)
        time.sleep(0.15)

    json.dump(short_descs, open(OUT_PATH, 'w'), indent=2, ensure_ascii=False)
    if tmp_path.exists():
        tmp_path.unlink()

    # --- Stats + examples ---
    lengths = [len(v.split()) for v in short_descs.values()]
    print(f'\nWrote {OUT_PATH}: {len(short_descs)} tags')
    print(f'  word count -- min={min(lengths)}  max={max(lengths)}  mean={sum(lengths)/len(lengths):.1f}')
    print('\nFive examples:')
    for k in list(short_descs)[:5]:
        print(f'  {k:30s} {short_descs[k]}')


Got 433 unique tags from in-memory items.
DeepSeek API key (or paste literal): ··········


DeepSeek (short):   0%|          | 0/433 [00:00<?, ?it/s]


Wrote tag_descriptions_short.json: 433 tags
  word count -- min=9  max=14  mean=10.9

Five examples:
  1980s                          retro pixel art, synthwave soundtrack, neon aesthetics, arcade-style gameplay, cold war themes
  1990's                         retro pixel graphics, chiptune music, limited saves, nostalgic gameplay, era-specific design
  2.5D                           side-scrolling gameplay, 3D-rendered sprites, depth illusion, platforming or action in layered environments
  2D                             side-scrolling platforming, pixel art, flat plane movement, retro aesthetic, limited depth
  2D Fighter                     side-view combat, special moves, combos, versus mode, precise inputs, character roster


## 3. Build the three text variants per game

*A = title only · B = title + tag list · C = title + LLM descriptions*

In [10]:
    # Build the three text variants per game
    # - A: title only
    # - B: title + tag-name list (template)
    # - C: title + structured short descriptions (one tag per phrase)
    #
    # The new C format uses SHORT descriptions and puts the tag KEYWORD first in each phrase,
    # so even after truncation the genre signal survives.

    TAGS_PER_GAME = 5

    # Prefer the short descriptions; fall back to the original long ones if needed.
    from pathlib import Path
    import json as _json
    _short_path = Path('tag_descriptions_short.json')
    if _short_path.exists():
        tag_descriptions_short = _json.load(open(_short_path))
        print(f'Using {_short_path} for condition C  ({len(tag_descriptions_short)} tags, '
              f'avg {sum(len(v.split()) for v in tag_descriptions_short.values())/len(tag_descriptions_short):.1f} words)')
        _C_SOURCE = tag_descriptions_short
    else:
        print('WARNING: tag_descriptions_short.json not found, falling back to long descriptions.')
        _C_SOURCE = tag_descriptions

    def text_A(row):
        return str(row['title']).strip()

    def text_B(row):
        tags = row['tags'][:TAGS_PER_GAME] if isinstance(row['tags'], (list, np.ndarray)) else []
        if len(tags) == 0:
            return str(row['title']).strip()
        return f"{row['title']} [SEP] This game's tags are: {', '.join(tags)}"

    def text_C(row):
        tags = row['tags'][:TAGS_PER_GAME] if isinstance(row['tags'], (list, np.ndarray)) else []
        if len(tags) == 0: # Changed from 'if not tags:'
            return str(row['title']).strip()
        # KEY DESIGN CHANGE: keep tag names visible inline -> 'Action (combat, fast pacing); RPG (leveling, ...)'
        # This way even if the encoder only sees the first half, the discriminative keywords are still there.
        pieces = []
        for t in tags:
            d = _C_SOURCE.get(t, '').strip()
            if d:
                pieces.append(f"{t} ({d})")
            else:
                pieces.append(t)
        return f"{row['title']} [SEP] {'; '.join(pieces)}"

    items['text_A'] = items.apply(text_A, axis=1)
    items['text_B'] = items.apply(text_B, axis=1)
    items['text_C'] = items.apply(text_C, axis=1)

    for col in ['text_A', 'text_B', 'text_C']:
        print(f'  {col}: mean len {items[col].str.len().mean():.0f} chars, max {items[col].str.len().max()}')

    print('\nExample game:', items.iloc[0]['title'])
    for col in ['text_A', 'text_B', 'text_C']:
        print(f'\n  {col}:')
        print(f'    {items.iloc[0][col][:300]}')

Using tag_descriptions_short.json for condition C  (433 tags, avg 10.9 words)
  text_A: mean len 19 chars, max 81
  text_B: mean len 100 chars, max 162
  text_C: mean len 560 chars, max 657

Example game: Prince of Persia: Warrior Within™

  text_A:
    Prince of Persia: Warrior Within™

  text_B:
    Prince of Persia: Warrior Within™ [SEP] This game's tags are: Action, Adventure, Parkour, Third Person, Great Soundtrack

  text_C:
    Prince of Persia: Warrior Within™ [SEP] Action (fast-paced combat, real-time controls, dodging, combos, reflex-based gameplay, direct player input); Adventure (exploration, puzzle solving, inventory management, narrative focus, point-and-click or 3D); Parkour (fast-paced movement, wall-running, vaul


### Diagnostic — token-length comparison for the three conditions



In [13]:
# Token-length check (run AFTER cell that builds text_A/B/C)
from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')
import numpy as np

MAX_LEN = 135 # Increased MAX_LEN to accommodate text_C better (p95 was 134)

for col in ['text_A', 'text_B', 'text_C']:
    lens = [len(_tok(s, add_special_tokens=True)['input_ids']) for s in items[col].tolist()]
    lens = np.array(lens)
    pct_trunc = (lens > MAX_LEN).mean() * 100
    print(f'{col}: mean={lens.mean():5.1f} tokens   p95={np.percentile(lens, 95):5.0f}   '
          f'max={lens.max():3d}   truncated={pct_trunc:5.1f}%  (MAX_LEN={MAX_LEN})')

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


text_A: mean=  6.6 tokens   p95=   12   max= 23   truncated=  0.0%  (MAX_LEN=96)
text_B: mean= 26.8 tokens   p95=   33   max= 43   truncated=  0.0%  (MAX_LEN=96)
text_C: mean=121.1 tokens   p95=  134   max=147   truncated= 99.3%  (MAX_LEN=96)


## 4. Tokenize all three conditions once

In [14]:
from transformers import AutoTokenizer

PRETRAINED = 'distilbert-base-uncased'
MAX_LEN    = 135   # bumped from 64 to 135 so condition C's structured descriptions fit comfortably

tokenizer = AutoTokenizer.from_pretrained(PRETRAINED)

# Map app_id -> compact index 0..N-1. Reserve index N for PAD-game.
app_ids = items['app_id'].tolist()
app_id_to_idx = {a: i for i, a in enumerate(app_ids)}
N_ITEMS = len(app_ids)
PAD_INDEX = N_ITEMS
print(f'N_ITEMS={N_ITEMS}, PAD_INDEX={PAD_INDEX}')

TOKENIZED = {}
for cond, col in [('A','text_A'), ('B','text_B'), ('C','text_C')]:
    enc = tokenizer(items[col].tolist(), padding='max_length', truncation=True,
                    max_length=MAX_LEN, return_tensors='pt')
    # add a PAD-game row of zeros at index N_ITEMS
    pad_ids  = torch.zeros(1, MAX_LEN, dtype=enc['input_ids'].dtype)
    pad_mask = torch.zeros(1, MAX_LEN, dtype=enc['attention_mask'].dtype)
    TOKENIZED[cond] = {
        'input_ids':      torch.cat([enc['input_ids'],  pad_ids],  dim=0),
        'attention_mask': torch.cat([enc['attention_mask'], pad_mask], dim=0),
    }
    print(f'  {cond}: tokenized shape {TOKENIZED[cond]["input_ids"].shape}')

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


N_ITEMS=3825, PAD_INDEX=3825
  A: tokenized shape torch.Size([3826, 96])
  B: tokenized shape torch.Size([3826, 96])
  C: tokenized shape torch.Size([3826, 96])


## 5. PyTorch Dataset



In [15]:
from torch.utils.data import Dataset, DataLoader

HISTORY_LEN = 50

def encode_history(hist_list):
    '''Convert raw app_ids to indices, left-pad with PAD_INDEX to HISTORY_LEN.'''
    if hist_list is None:
        idxs = []
    elif hasattr(hist_list, 'tolist'):
        idxs = [app_id_to_idx[a] for a in hist_list.tolist() if a in app_id_to_idx]
    else:
        idxs = [app_id_to_idx[a] for a in hist_list if a in app_id_to_idx]
    idxs = idxs[-HISTORY_LEN:]
    pad = [PAD_INDEX] * (HISTORY_LEN - len(idxs))
    return pad + idxs


# ----- training impressions: chunk every 5 rows (1 pos + 4 neg) -----
print('Building training groups (chunks of 5)...')
train_groups = []
n = len(train_samples)
for i in range(0, n - 4, 5):
    block = train_samples.iloc[i:i+5]
    labels = block['label'].tolist()
    # must be exactly [1, 0, 0, 0, 0]
    if labels != [1, 0, 0, 0, 0]:
        continue
    train_groups.append({
        'history':    list(block.iloc[0]['history']),
        'candidates': block['candidate'].tolist(),  # [pos, neg, neg, neg, neg]
    })
print(f'  built {len(train_groups):,} training groups')


def build_eval_groups(samples_df):
    '''Each (user_id, date) tuple = one impression; keep only those with both classes.'''
    groups = []
    for (uid, date), block in samples_df.groupby(['user_id', 'date'], sort=False):
        if block['label'].nunique() < 2:
            continue
        groups.append({
            'user_id':    int(uid),
            'history':    list(block.iloc[0]['history']),
            'candidates': block['candidate'].tolist(),
            'labels':     block['label'].tolist(),
        })
    return groups

print('Building eval groups...')
val_groups  = build_eval_groups(val_samples)
test_groups = build_eval_groups(test_samples)
print(f'  val  impressions: {len(val_groups):,}')
print(f'  test impressions: {len(test_groups):,}')


class TrainDataset(Dataset):
    def __init__(self, groups):
        self.groups = groups
    def __len__(self):
        return len(self.groups)
    def __getitem__(self, i):
        g = self.groups[i]
        return {
            'history':    torch.tensor(encode_history(g['history']), dtype=torch.long),
            'candidates': torch.tensor([app_id_to_idx[a] for a in g['candidates']], dtype=torch.long),
        }


class EvalDataset(Dataset):
    def __init__(self, groups):
        self.groups = groups
    def __len__(self):
        return len(self.groups)
    def __getitem__(self, i):
        g = self.groups[i]
        return {
            'history':    torch.tensor(encode_history(g['history']), dtype=torch.long),
            'candidates': torch.tensor([app_id_to_idx[a] for a in g['candidates']], dtype=torch.long),
            'labels':     torch.tensor(g['labels'], dtype=torch.float),
            'cand_ids':   torch.tensor(g['candidates'], dtype=torch.long),
            'user_id':    g['user_id'],
        }

def eval_collate(batch):
    '''Eval candidate counts vary — keep as a Python list, one impression at a time.'''
    return batch


Building training groups (chunks of 5)...
  built 20,113 training groups
Building eval groups...
  val  impressions: 12,652
  test impressions: 12,098


## 6. NRMS model 

Same architecture as the paper.

In [16]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

class AdditiveAttention(nn.Module):
    def __init__(self, hidden_size, attn_size=200):
        super().__init__()
        self.proj  = nn.Linear(hidden_size, attn_size)
        self.query = nn.Linear(attn_size, 1, bias=False)
    def forward(self, x, mask=None):
        # x: (B, L, D), mask: (B, L) of 1/0
        u = torch.tanh(self.proj(x))
        scores = self.query(u).squeeze(-1)            # (B, L)
        if mask is not None:
            mask = mask.bool()
            all_masked = ~mask.any(dim=1)             # (B,)
            safe_mask = mask.clone()
            safe_mask[all_masked] = True              # avoid -inf row
            scores = scores.masked_fill(~safe_mask, float('-inf'))
        w = F.softmax(scores, dim=-1)
        out = (w.unsqueeze(-1) * x).sum(dim=1)        # (B, D)
        if mask is not None:
            out = out * (~all_masked).unsqueeze(-1).float()
        return out


class GameEncoder(nn.Module):
    def __init__(self, pretrained, hidden_size, attn_size=200):
        super().__init__()
        self.plm = AutoModel.from_pretrained(pretrained)
        ph = self.plm.config.hidden_size
        self.attn = AdditiveAttention(ph, attn_size)
        self.proj = nn.Linear(ph, hidden_size)
    def forward(self, input_ids, attention_mask):
        tok = self.plm(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        pooled = self.attn(tok, attention_mask)
        return self.proj(pooled)


class UserEncoder(nn.Module):
    def __init__(self, hidden_size, num_heads=8, attn_size=200):
        super().__init__()
        self.mha  = nn.MultiheadAttention(hidden_size, num_heads, batch_first=True)
        self.attn = AdditiveAttention(hidden_size, attn_size)
    def forward(self, hist_vecs, hist_mask):
        # hist_vecs: (B, H, D), hist_mask: (B, H)
        kpm = ~hist_mask.bool()
        all_pad = kpm.all(dim=1)
        if all_pad.any():
            kpm[all_pad, 0] = False
        attended, _ = self.mha(hist_vecs, hist_vecs, hist_vecs, key_padding_mask=kpm)
        return self.attn(attended, hist_mask)


class NRMS(nn.Module):
    def __init__(self, pretrained, hidden_size=256, num_heads=8):
        super().__init__()
        self.game_enc = GameEncoder(pretrained, hidden_size)
        self.user_enc = UserEncoder(hidden_size, num_heads)
        self.hidden_size = hidden_size

    def _encode(self, app_idx, tok_ids, tok_mask):
        shape = app_idx.shape
        flat = app_idx.reshape(-1)
        ids  = tok_ids[flat]
        mask = tok_mask[flat]
        v = self.game_enc(ids, mask)
        return v.reshape(*shape, self.hidden_size)

    def forward(self, history_idx, cand_idx, tok_ids, tok_mask):
        hv = self._encode(history_idx, tok_ids, tok_mask)
        hm = (history_idx != PAD_INDEX).float()
        uv = self.user_enc(hv, hm)
        cv = self._encode(cand_idx, tok_ids, tok_mask)
        return torch.bmm(cv, uv.unsqueeze(-1)).squeeze(-1)

print('NRMS classes defined.')


NRMS classes defined.


## 7. Per-impression metrics

In [17]:
from sklearn.metrics import roc_auc_score

def _dcg(rel, k):
    rel = np.asarray(rel)[:k]
    if rel.size == 0: return 0.0
    return float((rel / np.log2(np.arange(2, rel.size + 2))).sum())

def ndcg_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores))
    rel = np.asarray(labels)[order]
    ideal = np.sort(np.asarray(labels))[::-1]
    idcg = _dcg(ideal, k)
    return 0.0 if idcg == 0 else _dcg(rel, k) / idcg

def mrr_score(labels, scores):
    order = np.argsort(-np.asarray(scores))
    rel = np.asarray(labels)[order]
    for i, r in enumerate(rel, 1):
        if r > 0: return 1.0 / i
    return 0.0

def aggregate(all_labels, all_scores):
    aucs, mrrs, n5s, n10s = [], [], [], []
    for l, s in zip(all_labels, all_scores):
        if len(set(l)) < 2: continue
        try:
            aucs.append(roc_auc_score(l, s))
        except ValueError:
            continue
        mrrs.append(mrr_score(l, s))
        n5s.append(ndcg_at_k(l, s, 5))
        n10s.append(ndcg_at_k(l, s, 10))
    return {
        'AUC':           float(np.mean(aucs))  if aucs  else 0.0,
        'MRR':           float(np.mean(mrrs))  if mrrs  else 0.0,
        'nDCG@5':        float(np.mean(n5s))   if n5s   else 0.0,
        'nDCG@10':       float(np.mean(n10s))  if n10s  else 0.0,
        'n_impressions': len(aucs),
    }


## 8. Config + output directory + resume logic

In [18]:
from tqdm.auto import tqdm

CFG = {
    'epochs':           1,       # bump to 2-3 if you have time
    'batch_size':       8,
    'grad_accum_steps': 16,      # effective batch = 128
    'lr':               1e-4,
    'weight_decay':     1e-2,
    'warmup_pct':       0.1,
    'max_grad_norm':    1.0,
    'eval_batch_size':  4,
    'hidden_size':      256,
    'num_heads':        8,
    'seed':             42,
    # Optional: subset for a quick sanity run; set to None for full data
    'train_subset':     None,    # e.g. 10000 for a fast test
    'eval_subset':      None,    # e.g. 2000
}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Config:', CFG)

OUT_DIR = Path('nrms_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = OUT_DIR / 'results_nrms.json'
if RESULTS_PATH.exists():
    RESULTS = json.loads(RESULTS_PATH.read_text())
    print(f'Resuming. Already done: {list(RESULTS.keys())}')
else:
    RESULTS = {}
    print('Starting fresh.')

def save_results():
    RESULTS_PATH.write_text(json.dumps(RESULTS, indent=2))


Device: cuda
Config: {'epochs': 1, 'batch_size': 8, 'grad_accum_steps': 16, 'lr': 0.0001, 'weight_decay': 0.01, 'warmup_pct': 0.1, 'max_grad_norm': 1.0, 'eval_batch_size': 4, 'hidden_size': 256, 'num_heads': 8, 'seed': 42, 'train_subset': None, 'eval_subset': None}
Starting fresh.


## 9. Training / evaluation / recommendation export

In [19]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

@torch.no_grad()
def evaluate_model(model, groups, TOK_IDS, TOK_MASK, keep_recommendations=False):
    model.eval()
    all_labels, all_scores = [], []
    recs = []
    title_lookup = dict(zip(items['app_id'], items['title']))
    for g in tqdm(groups, desc='eval', leave=False):
        hist = torch.tensor(encode_history(g['history']), dtype=torch.long).unsqueeze(0).to(DEVICE)
        cand = torch.tensor([app_id_to_idx[a] for a in g['candidates']], dtype=torch.long).unsqueeze(0).to(DEVICE)
        logits = model(hist, cand, TOK_IDS, TOK_MASK)[0].cpu().numpy()
        labels = list(g['labels'])
        scores = logits.tolist()
        all_labels.append(labels)
        all_scores.append(scores)
        if keep_recommendations:
            order = np.argsort(-logits)
            top10 = order[:10]
            top_app = [int(g['candidates'][i]) for i in top10]
            top_t   = [title_lookup.get(a, '?') for a in top_app]
            top_s   = [float(logits[i]) for i in top10]
            held    = [int(g['candidates'][i]) for i, l in enumerate(labels) if l == 1]
            recs.append({
                'user_id':       g.get('user_id'),
                'top_k_app_ids': top_app,
                'top_k_titles':  top_t,
                'top_k_scores':  top_s,
                'held_out_positives': held,
                'rank_of_first_positive': next((rk + 1 for rk, i in enumerate(order) if labels[i] == 1), -1),
            })
    model.train()
    return aggregate(all_labels, all_scores), recs


def train_one_condition(cond_key, cond_name):
    print('\n' + '=' * 70)
    print(f'  Condition {cond_key}: {cond_name}')
    print('=' * 70)

    torch.manual_seed(CFG['seed'])
    np.random.seed(CFG['seed'])
    random.seed(CFG['seed'])

    TOK_IDS  = TOKENIZED[cond_key]['input_ids'].to(DEVICE)
    TOK_MASK = TOKENIZED[cond_key]['attention_mask'].to(DEVICE)

    model = NRMS(PRETRAINED, hidden_size=CFG['hidden_size'], num_heads=CFG['num_heads']).to(DEVICE)

    tg = train_groups[:CFG['train_subset']] if CFG['train_subset'] else train_groups
    vg = val_groups[:CFG['eval_subset']]    if CFG['eval_subset']  else val_groups
    teg = test_groups[:CFG['eval_subset']]  if CFG['eval_subset']  else test_groups
    print(f'  train={len(tg):,}  val={len(vg):,}  test={len(teg):,}')

    train_loader = DataLoader(TrainDataset(tg), batch_size=CFG['batch_size'], shuffle=True,
                              num_workers=2, drop_last=True, pin_memory=True)

    total_steps  = max(1, (len(train_loader) // CFG['grad_accum_steps']) * CFG['epochs'])
    warmup_steps = int(CFG['warmup_pct'] * total_steps)
    optim = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sched = get_linear_schedule_with_warmup(optim, warmup_steps, total_steps)

    # ---- training ----
    nan_count, step = 0, 0
    for epoch in range(CFG['epochs']):
        t0 = time.time()
        running_loss, running_correct, running_n = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{CFG["epochs"]}', leave=True)
        for batch in pbar:
            hist = batch['history'].to(DEVICE)
            cand = batch['candidates'].to(DEVICE)
            label = torch.zeros(hist.size(0), dtype=torch.long, device=DEVICE)
            logits = model(hist, cand, TOK_IDS, TOK_MASK)
            loss   = F.cross_entropy(logits, label) / CFG['grad_accum_steps']

            if not torch.isfinite(loss):
                nan_count += 1
                optim.zero_grad()
                continue

            loss.backward()
            step += 1
            if step % CFG['grad_accum_steps'] == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['max_grad_norm'])
                optim.step(); sched.step(); optim.zero_grad()

            bs = hist.size(0)
            running_loss    += loss.item() * CFG['grad_accum_steps'] * bs
            running_correct += (logits.argmax(dim=-1) == 0).sum().item()
            running_n       += bs
            pbar.set_postfix(loss=f'{running_loss/max(1,running_n):.4f}',
                             acc=f'{running_correct/max(1,running_n):.3f}',
                             nan=nan_count)
        print(f'  epoch {epoch+1} done in {time.time()-t0:.1f}s, nan-batches={nan_count}')
        val_metrics, _ = evaluate_model(model, vg, TOK_IDS, TOK_MASK, keep_recommendations=False)
        print(f'  val: {val_metrics}')

    # ---- test + recommendations ----
    print('  Running final test evaluation + recommendation export...')
    test_metrics, test_recs = evaluate_model(model, teg, TOK_IDS, TOK_MASK, keep_recommendations=True)
    print(f'  TEST: {test_metrics}')

    # ---- save checkpoint + recs ----
    ckpt = OUT_DIR / f'checkpoint_{cond_key}.pt'
    torch.save({'state_dict': model.state_dict(), 'cfg': CFG, 'condition': cond_key,
                'cond_name': cond_name, 'metrics': test_metrics}, ckpt)
    print(f'  saved {ckpt}')

    rec_path = OUT_DIR / f'recommendations_{cond_key}.json'
    rec_path.write_text(json.dumps(test_recs, indent=2))
    print(f'  saved {rec_path}  ({len(test_recs)} impressions)')

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return test_metrics


## 9b. FAST MODE — freeze DistilBERT + use less data


In [20]:
FAST_MODE = True   # set False for the slow paper-faithful run

if FAST_MODE:
    print('FAST MODE ON')

    # ----- 1) Less data, bigger batch, shorter history -----
    CFG['epochs']        = 2          # cheap now, so we can afford 2
    CFG['batch_size']    = 32
    CFG['lr']            = 5e-4       # higher LR since only small layers train
    CFG['train_subset']  = 20000      # take only the first 20k training groups
    CFG['eval_subset']   = 3000       # only evaluate on 3k test impressions
    HISTORY_LEN          = 20         # was 50

    # ----- 2) Re-define encode_history with the shorter length -----
    def encode_history(hist_list, _H=HISTORY_LEN):
        if hist_list is None:
            idxs = []
        elif hasattr(hist_list, 'tolist'):
            idxs = [app_id_to_idx[a] for a in hist_list.tolist() if a in app_id_to_idx]
        else:
            idxs = [app_id_to_idx[a] for a in hist_list if a in app_id_to_idx]
        idxs = idxs[-_H:]
        pad = [PAD_INDEX] * (_H - len(idxs))
        return pad + idxs

    # ----- 3) Frozen GameEncoder with pre-cached vectors -----
    class GameEncoderFast(nn.Module):
        '''DistilBERT is frozen. We compute every game's BERT vector once at the start
        of training, store it in `_cache`, and during training just look up and project.'''
        def __init__(self, pretrained, hidden_size, attn_size=200):
            super().__init__()
            self.plm = AutoModel.from_pretrained(pretrained)
            for p in self.plm.parameters():
                p.requires_grad = False
            self.plm.eval()
            ph = self.plm.config.hidden_size
            self.proj = nn.Linear(ph, hidden_size)
            self._cache = None

        @torch.no_grad()
        def build_cache(self, tok_ids, tok_mask, bs=128):
            device = next(self.plm.parameters()).device
            N = tok_ids.shape[0]
            self._cache = torch.zeros(N, self.plm.config.hidden_size, device=device)
            self.plm.eval()
            for s in tqdm(range(0, N, bs), desc='cache BERT', leave=False):
                e = min(s + bs, N)
                out  = self.plm(input_ids=tok_ids[s:e], attention_mask=tok_mask[s:e]).last_hidden_state
                m = tok_mask[s:e].float().unsqueeze(-1)
                self._cache[s:e] = (out * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)

        def encode_by_idx(self, idx):
            shape = idx.shape
            pooled = self._cache[idx.reshape(-1)]
            v = self.proj(pooled)
            return v.reshape(*shape, -1)

    # ----- 4) NRMS that uses the frozen encoder -----
    class NRMSFast(nn.Module):
        def __init__(self, pretrained, hidden_size=256, num_heads=8):
            super().__init__()
            self.game_enc = GameEncoderFast(pretrained, hidden_size)
            self.user_enc = UserEncoder(hidden_size, num_heads)
            self.hidden_size = hidden_size
        def forward(self, history_idx, cand_idx, tok_ids, tok_mask):
            hv = self.game_enc.encode_by_idx(history_idx)
            hm = (history_idx != PAD_INDEX).float()
            uv = self.user_enc(hv, hm)
            cv = self.game_enc.encode_by_idx(cand_idx)
            return torch.bmm(cv, uv.unsqueeze(-1)).squeeze(-1)

    NRMS = NRMSFast   # override the global so train_one_condition picks this up

    # ----- 5) Wrap train_one_condition to call build_cache after model creation -----
    _orig_train_one_condition = train_one_condition
    def train_one_condition_fast(cond_key, cond_name):
        # We need the cache built BEFORE training. Patch the model creation.
        # Simplest: call original, but it'll already use NRMSFast since we overrode NRMS.
        # The cache build needs TOK_IDS/TOK_MASK which the original creates internally.
        # Easiest path: duplicate the body of train_one_condition with build_cache injected.
        # To avoid that duplication, we just monkey-patch __init__ of NRMSFast to auto-cache.
        return _orig_train_one_condition(cond_key, cond_name)

    # ----- 6) Patch: auto-build cache the first time forward() is called -----
    _orig_forward = NRMSFast.forward
    def _auto_cache_forward(self, history_idx, cand_idx, tok_ids, tok_mask):
        if self.game_enc._cache is None:
            self.game_enc.build_cache(tok_ids, tok_mask)
        return _orig_forward(self, history_idx, cand_idx, tok_ids, tok_mask)
    NRMSFast.forward = _auto_cache_forward

    # ----- 7) Re-build training groups with new history length -----
    # (encode_history is now patched; TrainDataset will use it next time it's instantiated)

    # Recreate train_groups with the shorter history applied during encoding
    # (since encode_history is now patched, the next dataset instantiation will use HISTORY_LEN=20)

    print('Updated config:')
    print(f'  epochs       = {CFG["epochs"]}')
    print(f'  batch_size   = {CFG["batch_size"]}')
    print(f'  lr           = {CFG["lr"]}')
    print(f'  train_subset = {CFG["train_subset"]:,} (of {len(train_groups):,})')
    print(f'  eval_subset  = {CFG["eval_subset"]:,} (of {len(test_groups):,})')
    print(f'  history_len  = {HISTORY_LEN}')
    print(f'  BERT         = FROZEN (cached once per condition)')
else:
    print('FAST_MODE off — slow paper-faithful run')


FAST MODE ON
Updated config:
  epochs       = 2
  batch_size   = 32
  lr           = 0.0005
  train_subset = 20,000 (of 20,113)
  eval_subset  = 3,000 (of 12,098)
  history_len  = 20
  BERT         = FROZEN (cached once per condition)


## 10. Run all three conditions




In [21]:
CONDITIONS = [
    ('A', 'title only'),
    ('B', 'title + template'),
    ('C', 'title + LLM desc'),
]

for cond_key, cond_name in CONDITIONS:
    if cond_key in RESULTS:
        print(f'>> Skipping {cond_key} (already in results_nrms.json)')
        continue
    metrics = train_one_condition(cond_key, cond_name)
    RESULTS[cond_key] = {'condition_name': cond_name, **metrics}
    save_results()

print('\n' + '=' * 70)
print('  FINAL RESULTS')
print('=' * 70)
for k, v in RESULTS.items():
    print(f'  {k} ({v.get("condition_name",""):<20s})  '
          f'AUC={v["AUC"]:.4f}  MRR={v["MRR"]:.4f}  '
          f'nDCG@5={v["nDCG@5"]:.4f}  nDCG@10={v["nDCG@10"]:.4f}  '
          f'n={v["n_impressions"]}')



  Condition A: title only


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

  train=20,000  val=3,000  test=3,000


Epoch 1/2:   0%|          | 0/625 [00:00<?, ?it/s]

cache BERT:   0%|          | 0/30 [00:00<?, ?it/s]

...

  epoch 2 done in 4.9s, nan-batches=0


eval:   0%|          | 0/3000 [00:00<?, ?it/s]

  val: {'AUC': 0.7226332305413625, 'MRR': 0.34646757219192575, 'nDCG@5': 0.3477183236521059, 'nDCG@10': 0.42625817401443883, 'n_impressions': 3000}
  Running final test evaluation + recommendation export...


eval:   0%|          | 0/3000 [00:00<?, ?it/s]

  TEST: {'AUC': 0.7026655839028347, 'MRR': 0.32431034036879025, 'nDCG@5': 0.316735153107255, 'nDCG@10': 0.3990616599416792, 'n_impressions': 3000}
  saved nrms_outputs/checkpoint_C.pt
  saved nrms_outputs/recommendations_C.json  (3000 impressions)

  FINAL RESULTS
  A (title only          )  AUC=0.6300  MRR=0.2976  nDCG@5=0.2780  nDCG@10=0.3472  n=3000
  B (title + template    )  AUC=0.7005  MRR=0.3455  nDCG@5=0.3364  nDCG@10=0.4113  n=3000
  C (title + LLM desc    )  AUC=0.7027  MRR=0.3243  nDCG@5=0.3167  nDCG@10=0.3991  n=3000


## 11. Results table + improvement check

In [22]:
df = pd.DataFrame(RESULTS).T
df.index.name = 'condition'
df = df[['condition_name', 'AUC', 'MRR', 'nDCG@5', 'nDCG@10', 'n_impressions']]
df.to_csv(OUT_DIR / 'results_nrms.csv')
print('=== Steam NRMS — three conditions ===\n')
print(df.round(4).to_string())

if 'A' in RESULTS and 'B' in RESULTS and 'C' in RESULTS:
    a = RESULTS['A']['AUC']
    b = RESULTS['B']['AUC']
    c = RESULTS['C']['AUC']
    print(f'\nAUC improvement of LLM-desc (C) vs:')
    print(f'  title-only (A): {(c-a)/a*100:+.1f}%')
    print(f'  template   (B): {(c-b)/b*100:+.1f}%')


=== Steam NRMS — three conditions ===

             condition_name       AUC       MRR    nDCG@5   nDCG@10 n_impressions
condition                                                                        
A                title only  0.630012  0.297577  0.278001  0.347175          3000
B          title + template  0.700477  0.345461  0.336361  0.411337          3000
C          title + LLM desc  0.702666   0.32431  0.316735  0.399062          3000

AUC improvement of LLM-desc (C) vs:
  title-only (A): +11.5%
  template   (B): +0.3%


## 12. Inspect recommendations (sanity check)

In [23]:
for cond_key, cond_name in CONDITIONS:
    rec_path = OUT_DIR / f'recommendations_{cond_key}.json'
    if not rec_path.exists():
        continue
    recs = json.loads(rec_path.read_text())
    print('\n' + '=' * 70)
    print(f'  Condition {cond_key}: {cond_name}  ({len(recs)} test impressions)')
    print('=' * 70)
    for r in recs[:3]:
        held = r['held_out_positives']
        print(f"\n  user {r['user_id']}  (held-out positive = {held}, rank of pos = {r['rank_of_first_positive']})")
        for rank, (aid, title, score) in enumerate(zip(r['top_k_app_ids'], r['top_k_titles'], r['top_k_scores']), 1):
            marker = '  <-- HIT' if aid in held else ''
            print(f'    {rank:2d}. [{aid:>7}] score={score:+.3f}  {title[:60]}{marker}')



  Condition A: title only  (3000 test impressions)

  user 625  (held-out positive = [1954200], rank of pos = 14)
     1. [ 225320] score=+2.069  Tomb Raider III
     2. [ 960990] score=+1.480  Beyond: Two Souls
     3. [ 363970] score=+1.236  Clicker Heroes
     4. [ 250320] score=+1.201  The Wolf Among Us
     5. [ 942970] score=+1.166  Unheard - Voices of Crime
     6. [ 411830] score=+1.143  SENRAN KAGURA SHINOVI VERSUS
     7. [ 493900] score=+1.020  Dungeons 3
     8. [ 247910] score=+1.001  Sniper Elite: Nazi Zombie Army 2
     9. [2096600] score=+0.947  Crysis 2 Remastered
    10. [ 757480] score=+0.940  Broken Reality

  user 1723  (held-out positive = [2058180], rank of pos = 14)
     1. [ 889510] score=+1.180  SENRAN KAGURA Burst Re:Newal
     2. [1520470] score=+0.675  Soul Dossier
     3. [1104380] score=+0.569  The Room VR: A Dark Matter
     4. [ 748360] score=+0.234  MY HERO ONE'S JUSTICE
     5. [ 627270] score=+0.228  Injustice™ 2
     6. [ 391720] score=+0.151  Laye

## 13. Zip outputs and download

In [24]:
import shutil

# Copy the input tag descriptions in too, so the archive is fully self-contained
shutil.copy(TAG_DESC_PATH, OUT_DIR / 'tag_descriptions_deepseek.json')

archive_name = 'nrms_outputs'
shutil.make_archive(archive_name, 'zip', OUT_DIR)
print(f'Created {archive_name}.zip')
for f in Path('.').glob(f'{archive_name}.zip'):
    print(f'  {f.name}  {f.stat().st_size/1024/1024:.1f} MB')

print('\nFiles inside nrms_outputs/:')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name:40s} {f.stat().st_size/1024:8.1f} KB')


Created nrms_outputs.zip
  nrms_outputs.zip  708.2 MB

Files inside nrms_outputs/:
  checkpoint_A.pt                          261272.7 KB
  checkpoint_B.pt                          261272.7 KB
  checkpoint_C.pt                          261272.7 KB
  recommendations_A.json                     2616.3 KB
  recommendations_B.json                     2651.7 KB
  recommendations_C.json                     2659.9 KB
  results_nrms.csv                              0.3 KB
  results_nrms.json                             0.6 KB
  tag_descriptions_deepseek.json              162.2 KB


In [25]:
if IN_COLAB:
    from google.colab import files
    files.download(f'{archive_name}.zip')
else:
    print(f'Local run — your archive is at ./{archive_name}.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>